# Bài 4.2 · Feature extraction, linear probe và phân cụm

**CIFAR-10** gồm ảnh màu 32 × 32 thuộc 10 lớp (máy bay, ô tô, chim, mèo, hươu, chó, ếch, ngựa, tàu, xe tải). Notebook không train mạng nào. Ta chỉ đổi mỗi ảnh thành một vector bằng một encoder có sẵn, rồi hỏi: vector ấy dễ phân loại đến đâu?

1. So ba biểu diễn: pixel thô, đặc trưng ResNet-50 (ImageNet) và embedding ảnh của CLIP.
2. **Linear probe**: logistic regression trên vector đã đóng băng. **kNN** theo cosine.
3. **Phân cụm** k-means không dùng nhãn, đo xem cụm khớp lớp đến đâu.
4. Văn bản: pooling trung bình phải bỏ qua token padding.

Bật **GPU** và **Internet**. Chạy hết khoảng 6 phút.

In [ ]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.decomposition import PCA

torch.manual_seed(0)
import transformers
transformers.logging.set_verbosity_error()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('thiết bị:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
N_TRAIN, N_TEST = 5000, 2000

## 1. Dữ liệu

Tải CIFAR-10 từ Hugging Face, lấy ngẫu nhiên 5000 ảnh train và 2000 ảnh test, cố định seed. Ảnh giữ dạng PIL; mỗi encoder tự tiền xử lý theo hợp đồng của nó.

In [ ]:
from datasets import load_dataset
cifar = load_dataset('uoft-cs/cifar10')

class HFImages:
    # Bộ ảnh Hugging Face dùng như dataset torchvision: ds[i] trả (ảnh PIL, nhãn).
    def __init__(self, split):
        self.d, self.targets, self.classes = split, list(split['label']), split.features['label'].names
    def __len__(self):
        return len(self.d)
    def __getitem__(self, i):
        r = self.d[int(i)]
        return r['img'].convert('RGB'), r['label']

train_full, test_full = HFImages(cifar['train']), HFImages(cifar['test'])
rng = np.random.default_rng(0)
tr_idx = rng.permutation(len(train_full))[:N_TRAIN]
te_idx = rng.permutation(len(test_full))[:N_TEST]
classes = train_full.classes
y_tr = np.array(train_full.targets)[tr_idx]
y_te = np.array(test_full.targets)[te_idx]
print(len(tr_idx), 'ảnh train |', len(te_idx), 'ảnh test | số ảnh mỗi lớp ở train:', np.bincount(y_tr))
fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for k, ax in enumerate(axes):
    i = tr_idx[np.flatnonzero(y_tr == k)[0]]
    ax.imshow(train_full[i][0]); ax.set_title(classes[k], fontsize=8); ax.axis('off')
plt.show()

## 2. Ba cách biến ảnh thành vector

- **Pixel thô**: duỗi 32 × 32 × 3 thành 3072 số.
- **ResNet-50**: bỏ lớp cuối, lấy vector 2048 chiều sau global average pooling. Ảnh được phóng lên 224 cho khớp kích thước lúc model học.
- **CLIP ViT-B/32**: embedding ảnh 512 chiều, model học bằng cách ghép ảnh với chú thích chữ.

Model đặt `eval()` và chạy trong `inference_mode()`: không lưu gradient, không cập nhật BatchNorm.

In [ ]:
def pixels(ds, idx):
    return np.stack([np.asarray(ds[i][0], dtype=np.float32).ravel() / 255 for i in idx])

class Wrap(torch.utils.data.Dataset):
    def __init__(self, ds, idx, tf): self.ds, self.idx, self.tf = ds, idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, k): return self.tf(self.ds[self.idx[k]][0])

def embed(fn, ds, idx, tf, batch=128):
    out = []
    with torch.inference_mode():
        for x in DataLoader(Wrap(ds, idx, tf), batch, num_workers=2):
            out.append(fn(x.to(device)).float().cpu())
    return torch.cat(out).numpy()

w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
resnet = torchvision.models.resnet50(weights=w)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device).eval()
res_tf = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(w.transforms().mean, w.transforms().std)])

from transformers import CLIPModel, CLIPImageProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),
                     T.Normalize(proc.image_mean, proc.image_std)])
# transformers 4 trả thẳng tensor; transformers 5 trả một object, vector chiếu nằm ở pooler_output.
feat = lambda out: out if torch.is_tensor(out) else out.pooler_output
clip_fn = lambda x: feat(clip.get_image_features(pixel_values=x))

feats = {}
t = time.time(); feats['pixel thô'] = (pixels(train_full, tr_idx), pixels(test_full, te_idx))
feats['ResNet-50'] = (embed(resnet, train_full, tr_idx, res_tf), embed(resnet, test_full, te_idx, res_tf))
feats['CLIP ViT-B/32'] = (embed(clip_fn, train_full, tr_idx, clip_tf), embed(clip_fn, test_full, te_idx, clip_tf))
for k, (a, b) in feats.items():
    print(f'{k:14s} train {a.shape} | test {b.shape}')
print(f'{time.time() - t:.0f} giây')

## 3. Linear probe và kNN

Cùng một bộ phân loại tuyến tính cho cả ba biểu diễn; chỉ đặc trưng thay đổi. Chuẩn hóa từng cột bằng mean, std **của train** rồi mới áp sang test. kNN dùng cosine trên vector đã chia độ dài, $k = 10$.

In [ ]:
rows = []
for name, (a, b) in feats.items():
    t = time.time()
    sc = StandardScaler().fit(a)
    lr = LogisticRegression(max_iter=2000, C=0.5).fit(sc.transform(a), y_tr)
    acc_lr = (lr.predict(sc.transform(b)) == y_te).mean()
    na, nb = a / np.linalg.norm(a, axis=1, keepdims=True), b / np.linalg.norm(b, axis=1, keepdims=True)
    acc_knn = (KNeighborsClassifier(10).fit(na, y_tr).predict(nb) == y_te).mean()
    rows.append((name, a.shape[1], acc_lr, acc_knn))
    print(f'{name:14s} linear probe {acc_lr:.3f} | kNN {acc_knn:.3f} | {time.time() - t:.0f} giây')
probe = pd.DataFrame(rows, columns=['biểu diễn', 'số chiều', 'linear probe', 'kNN (k=10)']).round(3)
probe

## 4. Linear probe theo số ảnh có nhãn

Biểu diễn tốt thì cần ít nhãn. Học lại linear probe với 100, 500, 2000 và 5000 ảnh train, chấm trên cùng 2000 ảnh test.

In [ ]:
sizes = [100, 500, 2000, 5000]
curve = []
for name, (a, b) in feats.items():
    for n in sizes:
        sc = StandardScaler().fit(a[:n])
        lr = LogisticRegression(max_iter=2000, C=0.5).fit(sc.transform(a[:n]), y_tr[:n])
        curve.append((name, n, (lr.predict(sc.transform(b)) == y_te).mean()))
curve = pd.DataFrame(curve, columns=['biểu diễn', 'số ảnh train', 'acc'])
fig, ax = plt.subplots(figsize=(6, 4))
for name, g in curve.groupby('biểu diễn', sort=False):
    ax.plot(g['số ảnh train'], g['acc'], marker='o', label=name)
ax.set_xscale('log'); ax.set_xlabel('số ảnh train có nhãn'); ax.set_ylabel('accuracy test'); ax.grid(alpha=.3); ax.legend()
plt.show()
curve.pivot(index='số ảnh train', columns='biểu diễn', values='acc').round(3)

## 5. Phân cụm không dùng nhãn

k-means với 10 cụm trên vector đã chuẩn hóa độ dài. NMI và ARI đo mức cụm khớp với lớp thật: 1 là khớp hoàn toàn, quanh 0 là như chia ngẫu nhiên. Nhãn chỉ dùng để chấm, không dùng để phân cụm.

In [ ]:
rows = []
for name, (a, b) in feats.items():
    z = b / np.linalg.norm(b, axis=1, keepdims=True)
    lab = KMeans(10, n_init=10, random_state=0).fit_predict(z)
    rows.append((name, normalized_mutual_info_score(y_te, lab), adjusted_rand_score(y_te, lab)))
pd.DataFrame(rows, columns=['biểu diễn', 'NMI', 'ARI']).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (name, (a, b)) in zip(axes, feats.items()):
    p = PCA(2, random_state=0).fit_transform(StandardScaler().fit_transform(b))
    sc = ax.scatter(p[:, 0], p[:, 1], c=y_te, cmap='tab10', s=5)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
fig.legend(sc.legend_elements()[0], classes, loc='center right', fontsize=8)
plt.subplots_adjust(right=0.88); plt.show()

## 6. Văn bản: pooling phải bỏ qua padding

Encoder văn bản trả một vector cho **mỗi token**. Muốn một vector cho cả câu, ta lấy trung bình các token. Trong một batch, câu ngắn được đệm thêm token padding cho bằng câu dài; nếu tính trung bình cả padding thì vector của câu ngắn đổi theo câu dài nhất trong batch, dù câu ngắn không đổi một chữ nào.

In [ ]:
from transformers import AutoTokenizer, AutoModel
name = 'sentence-transformers/all-MiniLM-L6-v2'
tok = AutoTokenizer.from_pretrained(name)
enc = AutoModel.from_pretrained(name).to(device).eval()

def sentence_vec(texts, use_mask=True):
    b = tok(texts, padding=True, return_tensors='pt').to(device)
    with torch.inference_mode():
        h = enc(**b).last_hidden_state
    m = b['attention_mask'].unsqueeze(-1).float() if use_mask else torch.ones_like(h[..., :1])
    v = (h * m).sum(1) / m.sum(1)
    return torch.nn.functional.normalize(v, dim=1).cpu()

short = 'A cat sleeps on the sofa.'
long = 'The quarterly financial report shows that revenue grew in every region except the northern market.'
for use_mask in [True, False]:
    alone = sentence_vec([short], use_mask)[0]
    batched = sentence_vec([short, long], use_mask)[0]
    print(f'{"có mask" if use_mask else "không mask":10s} cosine(câu ngắn chạy riêng, câu ngắn chạy chung batch) = {float(alone @ batched):.4f}')